## 1. 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
from utils import reduce_memory_usage

# 1-1. 학습 데이터 로드
data_path = '../data/prep/k-pick_total_v2.csv'
data = pd.read_csv(data_path)
data = reduce_memory_usage(data)

print(f'전체 데이터 형태: {data.shape}')
print(f'컬럼 목록: {data.columns.tolist()}')
print(f'\n[정답(reordered) 분포]')
print(data['reordered'].value_counts())
print(f'재구매 비율: {data["reordered"].mean():.4f}')

In [ ]:
# 1-2. 사용하지 않는 피처 제외 및 정답 데이터 분리
# v2 데이터는 eval_set이 모두 prior이므로 전체 데이터를 학습에 사용
unused_cols = ['user_id', 'order_id', 'eval_set', 'product_id', 'reordered']
feature_cols = [c for c in data.columns if c not in unused_cols]

X = data[feature_cols].fillna(0)
y = data['reordered'].astype(int)

print(f'사용 피처 수: {len(feature_cols)}')
print(f'피처 목록: {feature_cols}')
print(f'\nX 형태: {X.shape}  /  y 형태: {y.shape}')

del data

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1-3. 스케일링 (정규화)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print('스케일링 완료')
print(X_scaled.describe().loc[['mean', 'std']].round(4).to_string())

In [ ]:
from sklearn.model_selection import train_test_split

# 1-4. 학습용(train) / 검증용(val) / 테스트(test) 데이터 분리 (7:2:1)
# Step1: 전체에서 10%를 테스트로 분리
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_scaled, y, test_size=0.1, random_state=42, stratify=y
)
# Step2: 나머지 90%에서 검증(22.2%) 분리 -> 전체 기준 7:2:1
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=2/9, random_state=42, stratify=y_train_val
)

total = len(y)
print('[데이터 분리 결과 (7:2:1)]')
print(f'학습(Train) : {len(y_train):>10,}건  ({len(y_train)/total:.1%})')
print(f'검증(Val)   : {len(y_val):>10,}건  ({len(y_val)/total:.1%})')
print(f'테스트(Test): {len(y_test):>10,}건  ({len(y_test)/total:.1%})')

pos_ratio = y_train.mean()
print(f'\n[학습 데이터 클래스 비율]')
print(f'  재구매(1): {pos_ratio:.4f}  /  미구매(0): {1-pos_ratio:.4f}')

## 2. 모델 객체 생성 및 학습

In [ ]:
import lightgbm as lgb

# 2-1. 모델 객체 생성
dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

# 클래스 불균형 보정 가중치 (미구매:재구매 비율)
scale_weight = round((y_train == 0).sum() / (y_train == 1).sum(), 2)

params = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'boosting_type'    : 'gbdt',
    'scale_pos_weight' : scale_weight,
    'learning_rate'    : 0.05,
    'num_leaves'       : 63,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'min_data_in_leaf' : 100,
    'seed'             : 42,
    'verbose'          : -1
}

print('[파라미터]')
for k, v in params.items():
    print(f'  {k:<20}: {v}')

In [ ]:
# 2-2. 모델 학습 시작
model = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

print(f'\n최적 반복 횟수: {model.best_iteration}')
print(f'최고 검증 AUC : {model.best_score["valid"]["auc"]:.6f}')

In [ ]:
from sklearn.metrics import accuracy_score

# 2-3. 학습 세트 / 검증 세트 정확도 확인
train_probs = model.predict(X_train)
val_probs   = model.predict(X_val)

train_acc = accuracy_score(y_train, (train_probs >= 0.5).astype(int))
val_acc   = accuracy_score(y_val,   (val_probs   >= 0.5).astype(int))

print('[정확도 확인 (임계값 0.5)]')
print(f'  학습 세트 정확도: {train_acc:.4f}')
print(f'  검증 세트 정확도: {val_acc:.4f}')
print(f'  차이(과적합 지표): {train_acc - val_acc:.4f}')

In [ ]:
# 2-4. 각 feature들의 계수 확인
# LightGBM은 선형 계수 대신 Feature Importance(Gain)를 사용
# Gain: 해당 피처가 분기에 기여한 손실 감소 총량 (클수록 재구매 예측에 큰 영향)
importance_gain  = model.feature_importance(importance_type='gain')
importance_split = model.feature_importance(importance_type='split')

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gain'   : importance_gain,
    'Split'  : importance_split
}).sort_values(by='Gain', ascending=False).reset_index(drop=True)

print('[피처별 Gain (재구매 예측 기여도 내림차순)]')
print(coef_df.to_string(index=False))

# 중복 Gain 값 확인 (계수 중복 여부)
dup_groups = (
    coef_df.groupby('Gain')['Feature']
    .apply(list)
    .reset_index()
)
dup_groups = dup_groups[dup_groups['Feature'].map(len) > 1]

print('\n[Gain 값이 중복된 피처 그룹 확인]')
if not dup_groups.empty:
    for _, row in dup_groups.iterrows():
        print(f'  Gain={row["Gain"]} -> 해당 피처: {row["Feature"]}')
else:
    print('  중복된 Gain 값을 가진 피처가 없습니다.')

In [ ]:
# 2-4. 각 feature들의 계수 확인
# LightGBM은 선형 계수 대신 Feature Importance(Gain)를 사용
# Gain: 해당 피처가 분기에 기여한 손실 감소 총량 (클수록 재구매 예측에 큰 영향)
importance_gain  = model.feature_importance(importance_type='gain')
importance_split = model.feature_importance(importance_type='split')

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gain'   : importance_gain,
    'Split'  : importance_split
}).sort_values(by='Gain', ascending=False).reset_index(drop=True)

print('[피처별 Gain (재구매 예측 기여도 내림차순)]')
print(coef_df.to_string(index=False))

# 중복 Gain 값 확인 (계수 중복 여부)
dup_groups = (
    coef_df.groupby('Gain')['Feature']
    .apply(list)
    .reset_index()
)
dup_groups = dup_groups[dup_groups['Feature'].map(len) > 1]

print('\n[Gain 값이 중복된 피처 그룹 확인]')
if not dup_groups.empty:
    for _, row in dup_groups.iterrows():
        print(f'  Gain={row["Gain"]} -> 해당 피처: {row["Feature"]}')
else:
    print('  중복된 Gain 값을 가진 피처가 없습니다.')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 2-5. 피처 중요도 시각화
plt.figure(figsize=(10, 6))
plt.title('LightGBM Feature Importance (Gain 기준)')
sns.barplot(x='Gain', y='Feature', data=coef_df, palette='viridis')
for i, v in enumerate(coef_df['Gain']):
    plt.text(v, i, f' {v:,.0f}', va='center', fontsize=9)
plt.xlabel('Importance (Total Gain)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 3. 성능 평가

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 3-1. 테스트 세트로 최적 임계값 탐색
test_probs = model.predict(X_test)
best_f1, best_threshold = 0, 0.5

print(f"{'임계값':<10} | {'F1-Score':<10}")
print('-' * 25)
for t in np.arange(0.1, 0.91, 0.05):
    preds = (test_probs >= t).astype(int)
    current_f1 = f1_score(y_test, preds)
    print(f'{t:<10.2f} | {current_f1:<10.4f}')
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = t

print('-' * 25)
print(f'최적 임계값: {best_threshold:.2f}  |  최고 F1-Score: {best_f1:.4f}')

In [ ]:
# 3-2. 정확도, F1-Score 등 결과 출력
final_preds = (test_probs >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, final_preds).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision   = tp / (tp + fp)
npv         = tn / (tn + fn)
prevalence  = (tp + fn) / (tp + tn + fp + fn)
det_rate    = tp / (tp + tn + fp + fn)
det_prev    = (tp + fp) / (tp + tn + fp + fn)
bal_acc     = (sensitivity + specificity) / 2

print('=' * 55)
print(f'  최종 성능 리포트 (테스트 세트, 임계값: {best_threshold:.2f})')
print('=' * 55)
print(f'  Accuracy             : {accuracy_score(y_test, final_preds):.4f}')
print(f'  F1-Score             : {best_f1:.4f}')
print()
print(f'  Sensitivity (Recall) : {sensitivity:.5f}  <- 재구매자 중 맞춘 비율')
print(f'  Specificity          : {specificity:.5f}  <- 미구매자 중 맞춘 비율')
print(f'  Pos Pred Value (PPV) : {precision:.5f}  <- 산다 예측 중 실제 구매 비율')
print(f'  Neg Pred Value (NPV) : {npv:.5f}')
print(f'  Prevalence           : {prevalence:.5f}')
print(f'  Detection Rate       : {det_rate:.5f}')
print(f'  Detection Prevalence : {det_prev:.5f}')
print(f'  Balanced Accuracy    : {bal_acc:.5f}')
print('-' * 55)
print()
print('[기본 분류 리포트]')
print(classification_report(y_test, final_preds, target_names=['미구매(0)', '재구매(1)']))